# Processing Data from MATLAB

**Input data:** `sv_and_trip/<fault_type>/<file>.csv`

**Output data:** `processed/<fault_type>/<file>.csv`

**Processing order:**
1. Trim the first 0.3 seconds
2. Remove the MATLAB artifact in zero-sequence currents and voltages during normal mode
3. Calculate current and voltage amplitudes and phase angles using a sliding DFT
4. Process and save each file

**Each output file contains:**

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
DATASET_DIRS = [
    (Path('../data/arc_fault/sv_and_trip'),    Path('../data/arc_fault/processed')),
    (Path('../data/bolted_fault/sv_and_trip'), Path('../data/bolted_fault/processed')),
]

FAULT_TYPES = ['AG','BG','CG','ABG','BCG','CAG','ABCG','AB','BC','CA','ABC']

SAMPLE_RATE        = 4000   # 80 x 50 Hz = 4000 samples per second
SAMPLES_PER_PERIOD = 80     # samples per period
STEP_DFT           = 40     # sliding DFT step = half-period (10 ms)
CUT_SECONDS        = 0.3    # trim the beginning
CUT_ROWS           = int(CUT_SECONDS * SAMPLE_RATE)  # = 1200 rows

COLUMNS_IN  = ['Ia','Ib','Ic','I0','Ua','Ub','Uc','U0','label']
SIGNALS     = ['Ia','Ib','Ic','I0','Ua','Ub','Uc','U0']

Feature extraction uses a normalized fundamental phasor algorithm based on a DFT sliding window of N = 80 samples, which is one period of the 50 Hz power frequency. This is similar to algorithms implemented in microprocessor relay protection and automation terminals according to IEC 61850-9-2.

In [ ]:
def sliding_dft(signal: np.ndarray, N: int = 80, step: int = 40) -> tuple[np.ndarray, np.ndarray]:
    """
    Calculates the fundamental harmonic amplitude and phase angle using a sliding DFT window.

    Algorithm:
      X1 = (2/N) * sum( x[k] * exp(-j*2*pi*k/N) ) for k=0..N-1
      Amplitude = |X1|
      Angle     = angle(X1), in radians

    Parameters:
      N    - window length in samples (one period = 80 at 4000 Hz / 50 Hz)
      step - window shift in samples (half-period = 40, which corresponds to 10 ms)

    The result has length (len(signal) - N) // step + 1.
    """
    n = len(signal)
    size = (n - N) // step + 1

    # DFT basis vector for the first harmonic, computed once.
    k = np.arange(N)
    basis = np.exp(-1j * 2 * np.pi * k / N)  # shape: (N,)

    amplitude = np.empty(size)
    phase     = np.empty(size)

    for i in range(size):
        start  = i * step
        window = signal[start : start + N]
        X1 = (2.0 / N) * np.dot(window, basis)
        amplitude[i] = np.abs(X1)
        phase[i]     = np.angle(X1)

    return amplitude, phase

In [ ]:
def process_file(filepath: Path) -> pd.DataFrame | None:
    """Process one file."""
    # Load
    try:
        df = pd.read_csv(filepath, header=None, names=COLUMNS_IN, sep=';')
    except Exception:
        try:
            df = pd.read_csv(filepath, header=None, names=COLUMNS_IN)
        except Exception as e:
            print(f"  Read error {filepath.name}: {e}")
            return None

    # Trim the first 0.3 seconds.
    df = df.iloc[CUT_ROWS:].reset_index(drop=True)

    if len(df) < SAMPLES_PER_PERIOD:
        print(f"  File is too short after trimming: {filepath.name}")
        return None

    # Fix the Matlab artifact where I0 and U0 equal -1 during normal mode.
    # After trimming, the fault starts at row 600 (0.45s - 0.3s = 0.15s).
    fault_start_row = int(0.15 * SAMPLE_RATE)
    df.loc[df.index < fault_start_row, 'I0'] = df.loc[df.index < fault_start_row, 'I0'].replace(-1, 0.0)
    df.loc[df.index < fault_start_row, 'U0'] = df.loc[df.index < fault_start_row, 'U0'].replace(-1, 0.0)

    # Sliding DFT for each signal with STEP_DFT stride.
    result = {}
    for sig in SIGNALS:
        signal_values = df[sig].values.astype(float)
        amp, phi = sliding_dft(signal_values, N=SAMPLES_PER_PERIOD, step=STEP_DFT)
        result[f'amp_{sig}'] = amp
        result[f'phi_{sig}'] = phi

    # Label of the last sample in each DFT window.
    # Window i ends at sample i*STEP_DFT + SAMPLES_PER_PERIOD - 1.
    result['label'] = df['label'].values[SAMPLES_PER_PERIOD - 1::STEP_DFT].astype(int)

    df_out = pd.DataFrame(result)
    return df_out

In [ ]:
def process_all():
    '''Function for processing all files from all datasets'''
    total_files  = 0
    total_errors = 0

    for input_dir, output_dir in DATASET_DIRS:
        print(f"\n=== Processing: {input_dir} ===")

        for fault_type in FAULT_TYPES:
            in_folder  = input_dir  / fault_type
            out_folder = output_dir / fault_type

            if not in_folder.exists():
                print(f"\n[!] Folder not found: {in_folder}")
                continue

            out_folder.mkdir(parents=True, exist_ok=True)
            csv_files = sorted(in_folder.glob('*.csv'))
            print(f"[{fault_type}] Files: {len(csv_files)}")

            for fpath in csv_files:
                df_out = process_file(fpath)

                if df_out is None:
                    total_errors += 1
                    continue

                out_path = out_folder / fpath.name
                df_out.to_csv(out_path, index=False, sep=';')
                total_files += 1

            print(f"Total saved: {total_files} files")

    print(f"\nDone! Files processed: {total_files}")
    print(f"Errors: {total_errors}")

In [ ]:
if __name__ == '__main__':
    process_all()